# RetinAI CNN v2 — clasificador multifuente real

Esta es la libreta consolidada para las cuatro clases: **normal, retinopatía diabética, glaucoma y catarata**.

Fuentes independientes utilizadas: ODIR original, Sylhet depurado, remanente único del dataset de cuatro clases, IDRiD y RIM-ONE DL. Las copias CLAHE y preprocesadas no se cuentan como pacientes nuevos. IDRiD testing y RIM-ONE test por hospital permanecen fuera del entrenamiento.

Ejecuta todo con **GPU T4**. La descarga total puede tardar varios minutos. El modelo y el informe se guardan en `Mi unidad/RETINAI_CNN_V2/`.

In [ ]:
%pip -q install kagglehub gdown pandas scikit-learn importlib-resources
import tensorflow as tf
print('GPU:', tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('Selecciona GPU T4 en Entorno de ejecución > Cambiar tipo de entorno.')


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
destino = Path('/content/drive/MyDrive/RETINAI_CNN_V2')
requeridos = [
    destino / 'entrenar_clasificador_multifuente_real_v2.py',
    destino / 'retinai_fuentes_v2.csv',
    destino / 'odir_manifest_v2_split.csv',
    destino / 'sylhet_manifest_v2.csv',
]
faltan = [str(ruta) for ruta in requeridos if not ruta.is_file()]
if faltan:
    raise RuntimeError(f'Faltan archivos auxiliares en Drive: {faltan}')
print('Archivos auxiliares verificados.')


In [ ]:
import subprocess, sys
script = destino / 'entrenar_clasificador_multifuente_real_v2.py'
subprocess.run([sys.executable, str(script), str(destino)], check=True)


In [ ]:
import json
informe = destino / 'informe_clasificador_multifuente_real_v2.json'
datos = json.loads(informe.read_text(encoding='utf-8'))
print(json.dumps(datos['evaluations'], ensure_ascii=False, indent=2))
print('CRITERIOS:', json.dumps(datos['acceptance_criteria'], ensure_ascii=False, indent=2))
print('DECISIÓN FINAL:', datos['decision'])
print('Keras candidato:', datos['keras_candidate'])
print('TFLite:', datos['tflite_exported'] or 'NO GENERADO PORQUE EL MODELO FUE RECHAZADO')
print('Informe:', informe)
from google.colab import files
files.download(str(informe))


## Criterio de decisión

No evalúes el modelo por la exactitud de entrenamiento. Revisa especialmente `idrid_external_dr` y `rim_one_external_glaucoma`, porque representan cambios reales de fuente. Envía el archivo JSON generado para decidir si el candidato puede entrar en la demostración.